# Targeted F1 Boost

Root cause of F1 ceiling (~0.61): **5019 features vs 2545 training samples** → severe overfitting.

Strategy:
1. Feature selection via XGBoost importance (top 50 / 100 / 200)
2. Hyperparameter search **scoring=f1** on reduced feature set
3. Threshold optimization on final model
4. Stacking ensemble as final step

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, make_scorer)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
%matplotlib inline

SEED = 42
np.random.seed(SEED)

## 1. Load Data

In [ ]:
feature_dir = Path('../data/features')

X_train = pd.read_pickle(feature_dir / 'X_train_temporal.pkl')
X_test  = pd.read_pickle(feature_dir / 'X_test_temporal.pkl')
y_train = pd.read_pickle(feature_dir / 'y_train_cls_temporal.pkl')
y_test  = pd.read_pickle(feature_dir / 'y_test_cls_temporal.pkl')

print(f"Train: {X_train.shape}  positive rate: {y_train.mean()*100:.1f}%")
print(f"Test:  {X_test.shape}   positive rate: {y_test.mean()*100:.1f}%")
print(f"\nFeature/sample ratio: {X_train.shape[1]/X_train.shape[0]:.1f}x  (problem: >> 1)")

## 2. Feature Selection via XGBoost Importance

Train a quick XGBoost to rank features, then keep the top K.

In [ ]:
# Quick XGBoost to get feature importances
selector_xgb = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.5,
    scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
    random_state=SEED, n_jobs=-1, verbosity=0, eval_metric='logloss'
)
selector_xgb.fit(X_train, y_train)

importances = pd.Series(selector_xgb.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False)

print("Top 30 features by XGBoost importance:")
print(importances.head(30).to_string())

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
importances.head(50).plot(kind='barh', ax=ax)
ax.set_title('Top 50 XGBoost Feature Importances')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Compare F1 across feature subset sizes
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
f1_scorer = make_scorer(f1_score)

k_values = [50, 100, 200, 500, 1000]
k_results = {}

for k in k_values:
    top_k_features = importances.head(k).index.tolist()
    X_tr_k = X_train[top_k_features]
    X_te_k = X_test[top_k_features]

    model_k = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
        random_state=SEED, n_jobs=-1, verbosity=0, eval_metric='logloss'
    )
    cv_f1 = cross_val_score(model_k, X_tr_k, y_train, cv=cv, scoring='f1').mean()

    model_k.fit(X_tr_k, y_train)
    probs = model_k.predict_proba(X_te_k)[:, 1]
    # Optimize threshold
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.2, 0.65, 0.01):
        f = f1_score(y_test, probs >= t)
        if f > best_f1:
            best_f1, best_t = f, t

    k_results[k] = {'cv_f1': cv_f1, 'test_f1': best_f1, 'threshold': best_t}
    print(f"k={k:5d}: CV F1={cv_f1:.4f}  Test F1={best_f1:.4f} (t={best_t:.2f})")

best_k = max(k_results, key=lambda k: k_results[k]['test_f1'])
print(f"\nBest k={best_k}: Test F1={k_results[best_k]['test_f1']:.4f}")

## 3. Hyperparameter Tuning on Best Feature Subset (scoring=F1)

In [ ]:
# Use best_k features found above
top_features = importances.head(best_k).index.tolist()
X_tr = X_train[top_features]
X_te = X_test[top_features]

pos_weight = (1 - y_train.mean()) / y_train.mean()

param_dist = {
    'n_estimators':     [300, 500, 700, 1000],
    'max_depth':        [3, 4, 5, 6],
    'learning_rate':    [0.01, 0.03, 0.05, 0.08, 0.1],
    'subsample':        [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5, 7, 10],
    'reg_lambda':       [0.5, 1.0, 2.0, 5.0],
    'reg_alpha':        [0.0, 0.1, 0.5, 1.0],
    'gamma':            [0, 0.1, 0.2, 0.5],
}

base_xgb = XGBClassifier(
    scale_pos_weight=pos_weight,
    random_state=SEED, n_jobs=-1, verbosity=0, eval_metric='logloss'
)

search = RandomizedSearchCV(
    base_xgb, param_dist,
    n_iter=60,
    scoring='f1',          # <-- optimize F1 directly
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    n_jobs=-1, random_state=SEED, verbose=1
)
search.fit(X_tr, y_train)

print(f"Best CV F1 : {search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

In [ ]:
# Evaluate best model on test set
best_xgb = search.best_estimator_
probs_tuned = best_xgb.predict_proba(X_te)[:, 1]

# Threshold sweep
best_f1_tuned, best_t_tuned = 0, 0.5
for t in np.arange(0.20, 0.65, 0.005):
    f = f1_score(y_test, probs_tuned >= t)
    if f > best_f1_tuned:
        best_f1_tuned, best_t_tuned = f, t

y_pred_tuned = (probs_tuned >= best_t_tuned).astype(int)
print(f"Tuned XGBoost (top {best_k} features):")
print(f"  Best threshold : {best_t_tuned:.3f}")
print(f"  F1             : {best_f1_tuned:.4f}")
print(f"  Precision      : {precision_score(y_test, y_pred_tuned):.4f}")
print(f"  Recall         : {recall_score(y_test, y_pred_tuned):.4f}")
print(f"  ROC-AUC        : {roc_auc_score(y_test, probs_tuned):.4f}")

## 4. Stacking Ensemble on Best Feature Subset

In [ ]:
# Base learners use the tuned feature subset
base_learners = [
    ('xgb', XGBClassifier(
        **search.best_params_,
        scale_pos_weight=pos_weight,
        random_state=SEED, n_jobs=-1, verbosity=0, eval_metric='logloss'
    )),
    ('lgbm', LGBMClassifier(
        n_estimators=500, num_leaves=31, learning_rate=0.05,
        min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0, class_weight='balanced',
        random_state=SEED, n_jobs=-1, verbose=-1
    )),
    ('lr', LogisticRegression(
        max_iter=3000, solver='saga', class_weight='balanced',
        random_state=SEED, n_jobs=-1
    )),
]

meta_learner = LogisticRegression(max_iter=1000, random_state=SEED)

stacker = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    stack_method='predict_proba',
    n_jobs=-1
)

print("Training stacking ensemble...")
stacker.fit(X_tr, y_train)
print("Done.")

probs_stack = stacker.predict_proba(X_te)[:, 1]

best_f1_stack, best_t_stack = 0, 0.5
for t in np.arange(0.20, 0.65, 0.005):
    f = f1_score(y_test, probs_stack >= t)
    if f > best_f1_stack:
        best_f1_stack, best_t_stack = f, t

y_pred_stack = (probs_stack >= best_t_stack).astype(int)
print(f"\nStacking Ensemble (top {best_k} features):")
print(f"  Best threshold : {best_t_stack:.3f}")
print(f"  F1             : {best_f1_stack:.4f}")
print(f"  Precision      : {precision_score(y_test, y_pred_stack):.4f}")
print(f"  Recall         : {recall_score(y_test, y_pred_stack):.4f}")
print(f"  ROC-AUC        : {roc_auc_score(y_test, probs_stack):.4f}")

## 5. LightGBM with Dart Boosting (often better on sparse features)

In [ ]:
dart_lgbm = LGBMClassifier(
    boosting_type='dart',
    n_estimators=300,
    num_leaves=31,
    learning_rate=0.05,
    drop_rate=0.1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    class_weight='balanced',
    random_state=SEED, n_jobs=-1, verbose=-1
)

dart_lgbm.fit(X_tr, y_train)
probs_dart = dart_lgbm.predict_proba(X_te)[:, 1]

best_f1_dart, best_t_dart = 0, 0.5
for t in np.arange(0.20, 0.65, 0.005):
    f = f1_score(y_test, probs_dart >= t)
    if f > best_f1_dart:
        best_f1_dart, best_t_dart = f, t

y_pred_dart = (probs_dart >= best_t_dart).astype(int)
print(f"LightGBM DART (top {best_k} features):")
print(f"  Best threshold : {best_t_dart:.3f}")
print(f"  F1             : {best_f1_dart:.4f}")
print(f"  Precision      : {precision_score(y_test, y_pred_dart):.4f}")
print(f"  Recall         : {recall_score(y_test, y_pred_dart):.4f}")
print(f"  ROC-AUC        : {roc_auc_score(y_test, probs_dart):.4f}")

## 6. Final Comparison

In [ ]:
baseline_f1 = 0.6137  # nb30 best (XGBoost default threshold)

results = [
    {'Model': 'Baseline (nb30 XGBoost, all features)', 'F1': baseline_f1, 'ROC-AUC': 0.8095},
    {'Model': f'XGBoost top-{best_k} features (no tuning)', 'F1': k_results[best_k]['test_f1'], 'ROC-AUC': None},
    {'Model': f'XGBoost top-{best_k} + RandomizedSearch (F1)', 'F1': best_f1_tuned, 'ROC-AUC': roc_auc_score(y_test, probs_tuned)},
    {'Model': f'Stacking top-{best_k}', 'F1': best_f1_stack, 'ROC-AUC': roc_auc_score(y_test, probs_stack)},
    {'Model': f'LightGBM DART top-{best_k}', 'F1': best_f1_dart, 'ROC-AUC': roc_auc_score(y_test, probs_dart)},
]

results_df = pd.DataFrame(results).sort_values('F1', ascending=False)
print("=" * 70)
print("FINAL F1 COMPARISON")
print("=" * 70)
print(results_df.to_string(index=False))

best = results_df.iloc[0]
delta = best['F1'] - baseline_f1
print(f"\n*** BEST: {best['Model']} — F1={best['F1']:.4f} ({delta:+.4f} vs baseline) ***")

## 7. Save Best Model

In [ ]:
# Save best model + feature list + threshold
best_model_name = results_df.iloc[0]['Model']

if 'Stacking' in best_model_name:
    best_model_obj = stacker
    best_threshold = best_t_stack
elif 'DART' in best_model_name:
    best_model_obj = dart_lgbm
    best_threshold = best_t_dart
else:
    best_model_obj = best_xgb
    best_threshold = best_t_tuned

model_dir = Path('../models/classification')
model_dir.mkdir(parents=True, exist_ok=True)

artifact = {
    'model': best_model_obj,
    'features': top_features,
    'threshold': best_threshold,
    'test_f1': results_df.iloc[0]['F1'],
}

with open(model_dir / 'best_f1_model.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print(f"Saved best model: {best_model_name}")
print(f"  Features : {len(top_features)}")
print(f"  Threshold: {best_threshold:.3f}")
print(f"  Test F1  : {results_df.iloc[0]['F1']:.4f}")